# RQ2-0 - the segmentation ceiling

Stage attribution follows the order in which the pipeline runs, so the first thing that can
break is **segmentation**, not retrieval. This notebook measures, for every chunking method and
every answerable question, the best the segmentation still allows:

> **ceiling** = the fraction of the gold span covered by the union of the **five best
> segments of the gold document** under that method.

Five, because five segments are what the context budget passes to the generator. If the ceiling
is below 0.80, no retriever could have delivered the evidence: the span did not survive
segmentation, and the question is **S1 segmentation loss**. Retrieval is judged only on the
questions where the ceiling allowed success.

Nothing here uses a ranking, a score or an answer - only the cached segment tables of notebook
00 and the official gold spans.

### No GPU needed
**Runtime -> Change runtime type -> CPU.** It reads the cached parquet tables and takes about
a minute.

## 1. Open the cached segment tables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
PROJECT = '/content/drive/MyDrive/techqa_rq1'
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
print('project folder:', PROJECT)

In [ ]:
import sys, glob, json, re
import pandas as pd, numpy as np
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
import rq1_core as C
C.setup(project_dir=PROJECT, quiet=True)      # the cache folder carries the run tag
CACHE, RESULTS = str(C.PATHS.CACHE), str(C.PATHS.OUT)
paths = sorted(glob.glob(f'{CACHE}/chunks_*.parquet'))
if not paths:                                  # a different run tag on this Drive
    folders = sorted(glob.glob(f'{PROJECT}/cache_*'),
                     key=lambda f: len(glob.glob(f + '/chunks_*.parquet')), reverse=True)
    if folders and glob.glob(folders[0] + '/chunks_*.parquet'):
        CACHE = folders[0]
        RESULTS = CACHE.replace('/cache_', '/outputs_')
        paths = sorted(glob.glob(f'{CACHE}/chunks_*.parquet'))
print('cache   ->', CACHE)
print('outputs ->', RESULTS)
tables = {}
for path in paths:
    method = re.sub(r'^chunks_|\.parquet$', '', os.path.basename(path))
    tables[method] = pd.read_parquet(path)
    print(f'{method:<26} {len(tables[method]):>8,} segments')
assert tables, (f'no chunks_*.parquet under {PROJECT} - run notebook 00 first, or point CACHE '
                'at the folder that holds them')

## 2. The questions and their official gold spans

The gold document and the character offsets come from the official annotation. They are read here for measurement only; no ranking in this study ever sees them.

In [ ]:
# The whole labelled dataset, on the fast local disk.
# Notebook 00 mirrors the four extracted files to Drive, so this fresh runtime restores them
# with a file copy (~1 min) instead of re-downloading and re-extracting the archive (~5 min).
# getattr() keeps this working whichever build of notebook 00 wrote the library to Drive.
if not C.dataset_present():
    if getattr(C, 'dataset_mirrored', lambda: False)():
        print('restoring the corpus from the Drive mirror ...')
        C.restore_dataset_from_mirror()
    else:
        dl, ex = C.download_commands()
        get_ipython().system(dl)
        get_ipython().system(ex)
        if hasattr(C, 'mirror_dataset'):
            C.mirror_dataset()
ds = C.load_dataset()
print(json.dumps(ds.summary(), indent=1))

In [ ]:
def _int(x):
    # a few questions carry an empty or non-numeric offset; they have no usable span
    try:
        return int(x)
    except (TypeError, ValueError):
        return None

gold = pd.DataFrame([
    {'question_id': q['QUESTION_ID'], 'split': q['SPLIT'], 'doc_id': str(q['DOCUMENT']),
     'start': _int(q['START_OFFSET']), 'end': _int(q['END_OFFSET'])}
    for q in ds.questions
    if (str(q.get('DOCUMENT', '')) not in ('', 'nan')
        and _int(q['START_OFFSET']) is not None and _int(q['END_OFFSET']) is not None
        and _int(q['END_OFFSET']) > _int(q['START_OFFSET']))
])
print(f'{len(gold):,} answerable questions with a gold span')
gold.head(3)

## 3. The ceiling

`union_coverage` is the same merge-intervals function the pipeline uses for gold-span coverage, so the ceiling and the measured coverage are directly comparable: coverage can never exceed the ceiling.

In [ ]:
from itertools import combinations
CONTEXT_K = C.CFG.CONTEXT_K            # five segments are passed to the generator
EXACT_LIMIT = 18                       # beyond this many overlapping segments, go greedy

def union_coverage(intervals, s, e):
    sel = sorted((max(a, s), min(b, e)) for a, b in intervals if b > s and a < e)
    merged = []
    for a, b in sel:
        if not merged or a > merged[-1][1]:
            merged.append([a, b])
        else:
            merged[-1][1] = max(merged[-1][1], b)
    return sum(b - a for a, b in merged) / max(1, e - s)

def best_union(hits, s, e, k):
    """The most of the span any k segments can carry.

    Picking the k segments with the largest individual overlap is NOT the answer: with
    sliding windows the k biggest can sit on top of each other at one end of a long span,
    while a different k spread across it covers more. That would put the ceiling below what
    retrieval actually delivered, so the maximum is taken over subsets - exactly while the
    number of overlapping segments is small, and by marginal gain in the rare wide case.
    """
    if len(hits) <= k:
        return union_coverage(hits, s, e)
    if len(hits) <= EXACT_LIMIT:
        return max(union_coverage(c, s, e) for c in combinations(hits, k))
    chosen, rest = [], list(hits)
    for _ in range(k):                 # union coverage is monotone and submodular
        pick = max(rest, key=lambda x: union_coverage(chosen + [x], s, e))
        chosen.append(pick)
        rest.remove(pick)
    return union_coverage(chosen, s, e)

rows = []
for method, tbl in tables.items():
    by_doc = {d: g[['char_start', 'char_end']].to_numpy()
              for d, g in tbl.groupby('doc_id', sort=False)}
    for q in gold.itertuples():
        segs = by_doc.get(q.doc_id)
        if segs is None or not len(segs):
            rows.append((method, q.question_id, q.split, 0.0, 0.0, 0))
            continue
        overlap = np.minimum(segs[:, 1], q.end) - np.maximum(segs[:, 0], q.start)
        hits = [(int(a), int(b)) for (a, b), o in zip(segs, overlap) if o > 0]
        single = max((union_coverage([h], q.start, q.end) for h in hits), default=0.0)
        rows.append((method, q.question_id, q.split,
                     best_union(hits, q.start, q.end, CONTEXT_K), single, len(segs)))

ceiling = pd.DataFrame(rows, columns=['method', 'question_id', 'split',
                                      'segmentation_ceiling', 'best_single_segment',
                                      'segments_in_gold_document'])
print(ceiling.groupby('method').segmentation_ceiling.agg(['mean', 'min',
      ('below_0.80', lambda s: float((s < 0.80).mean()))]).round(3).to_string())

## 4. Consistency with the measured coverage

The ceiling is an upper bound on what retrieval can deliver. If any question's measured `gold_span_coverage` exceeded its ceiling, the two would not be measuring the same thing, and the check below would fail.

In [ ]:
# The per-question traces are written with four decimals, so a measured coverage can sit up
# to 5e-5 above a ceiling it actually equals. Anything larger than that is a real disagreement.
TOL = 1e-4
viol = []
for name in ('A',):
    f = pd.read_csv(f'{RESULTS}/rq1_factor{name}_per_question.csv')
    m = f.merge(ceiling, on=['method', 'question_id'], how='inner')
    bad = m[m.gold_span_coverage > m.segmentation_ceiling + TOL]
    viol.append(bad)
    print(f'notebook {name}: {len(m):,} rows compared, {len(bad)} above the ceiling')
bad = pd.concat(viol) if viol else pd.DataFrame()
if len(bad):                            # show them before failing, so the cause is visible
    print(bad[['method', 'question_id', 'gold_span_coverage',
               'segmentation_ceiling']].head(10).to_string(index=False))
assert len(bad) == 0, 'measured coverage above the segmentation ceiling - the tables disagree'
print('the ceiling is an upper bound on every measured coverage')

## 5. Save it beside the traces

In [ ]:
out = f'{RESULTS}/rq2_segmentation_ceiling.csv'
ceiling.to_csv(out, index=False)
print('written:', out, len(ceiling), 'rows')
try:
    from google.colab import files
    files.download(out)
except Exception as e:
    print('download it from Drive instead:', e)